In [ ]:
import sys, os 
import geopandas as gpd 
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
data_path = os.path.join(os.getcwd(), '..', 'data', 'geojson', 'polygon_cleaned.geojson')
gdf = gpd.read_file(data_path)

In [2]:
gdf

,CODIGO,OBSERV,INSUMO,APOYO,ASIGNACION,Asignado,OAM,area,url_type,download_url,poly_id,image_uid,geometry
0,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,34429.374,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,0,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.97056 10.61903, -72.97054 ..."
1,21,seasonal crops,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,6784.769,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.96914 10.61884, -72.96915 ..."
2,112,urban not continuous,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,12227.436,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,2,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.96928 10.61948, -72.96931 ..."
3,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,19385.187,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,3,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.97264 10.61976, -72.97273 ..."
4,313,fragmented forest,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,5013.914,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,4,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.96917 10.62053, -72.9691 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1948,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,1528.937,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1948,img_0029_37df798b,"MULTIPOLYGON (((-75.123 3.80901, -75.12297 3.8..."
1949,333,Bare areas,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,0.026,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1949,img_0029_37df798b,"MULTIPOLYGON (((-75.12391 3.81075, -75.1239 3...."
1950,232,wooded pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,10536.989,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1950,img_0029_37df798b,"MULTIPOLYGON (((-75.12278 3.8104, -75.12282 3...."
1951,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,5139.198,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1951,img_0029_37df798b,"MULTIPOLYGON (((-75.12367 3.81079, -75.12357 3..."


### Build the COG URL based on the image_uid 

We need to build the COG URL so we can query the image stats for each polygon.

In [ ]:
base_url = os.path.join(os.getcwd(), '..', 'data', 'images', 'cog')
row = gdf.iloc[0]
cog_url = f"{base_url}/{row['image_uid']}.tif"
geom = [row.geometry.__geo_interface__]
cog_url

'/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../data/images/cog/img_0000_c4a77f3e.tif'

## Reprojection and computation 

First, we handle the reprojection issue. The drone images are in a local projected coordinate system of WGS84, but the geometry is in degrees following EPSG:4326, so we need to project the geometry into the CRS of the raster (it is easier this way than the other way around). After that, we take advantage of the tiles we constructed earlier with COG - we only fetch the part of the image for that polygon. Once we have that, we have the band data. We have 3 bands in the image.

We observed mainly two main artifacts in the drone images: 
- First: there are no-data pixels in the drone images, especially near the boundaries and sometimes randomly in the middle, mostly because of how the drone image was processed and collected - maybe there were no overlapping pixels.
- We also observed that the pixel values are 0 in random areas, possibly seeming like faulty pixels, or we can say invalid pixels because the values were not even in the 0 to 255 range. 

Hence, to handle them, we first used a function from numpy's masked array that helps us identify the invalid pixels, and then we filtered the nodata values in the raster, which helped us remove those no-data pixels. We will study in depth whether they would have any impact on the mean computations!

In [ ]:
import rasterio 
from rasterio.mask import mask
import numpy as np 
with rasterio.open(cog_url) as src:
    print("Raster CRS:", src.crs)
    print("Raster bounds:", src.bounds)
    
    print("\nOriginal geometry CRS:", gdf.crs)
    print("Original geometry bounds:", row.geometry.bounds)
    
    geom_gdf = gpd.GeoDataFrame([row], geometry='geometry', crs=gdf.crs)
    geom_reprojected = geom_gdf.to_crs(src.crs)
    geom_transformed = [geom_reprojected.geometry.iloc[0].__geo_interface__]
    
    print("\nTransformed bounds:", geom_reprojected.geometry.iloc[0].bounds)
    
    masked_data, _ = mask(src, geom_transformed, crop=True, all_touched=False)

    stats = {}
    for band_idx, band_name in enumerate(['r', 'g', 'b'], start=1):
        band_data = masked_data[band_idx - 1]
        
        is_masked = np.ma.isMaskedArray(band_data) 

        if is_masked:
            valid = band_data[~band_data.mask]  # data=[1,2,0,4], mask=[False,False,True,False] -> valid = [1,2,4]
        else:
            nodata_mask = band_data != src.nodata  # plain ndarray: [1, 2, -9999, 4], nodata = -9999 -> valid = [1,2,4]
            valid = band_data[nodata_mask]

        # We need to remove the noise here, maybe more robust approach?
        
        if valid.size > 0:
            stats[f'{band_name}_mean'] = float(np.mean(valid))
            stats[f'{band_name}_std'] = float(np.std(valid))
        else:
            stats[f'{band_name}_mean'] = np.nan
            stats[f'{band_name}_std'] = np.nan
    print(stats)

Raster CRS: EPSG:32618
Raster bounds: BoundingBox(left=721782.2054845318, bottom=1174560.63571348, right=722310.3216997313, top=1174796.2209395552)

Original geometry CRS: EPSG:4326
Original geometry bounds: (-72.9714814422654, 10.618840012053344, -72.96914035652021, 10.620975138959965)

Transformed bounds: (721925.7615761297, 1174560.7526012938, 722182.7255575805, 1174796.086679114)
{'r_mean': 67.11867219203522, 'r_std': 63.05077827901257, 'g_mean': 72.36945632182152, 'g_std': 65.61500049739803, 'b_mean': 50.0079449993589, 'b_std': 48.51453627072589}


### Build the features 

Here we try to extract different types of statistics from the image in our polygon. We have already calculated the area of the polygon that can be used as a feature, and here we are using mean, std, var in each band as well as some band indices, green index, and color profile!

In [4]:
from src.stats import compute_raster_stats

In [ ]:
gdf_with_stats = compute_raster_stats(gdf, base_url)

Processing polygon wrt rasters: 100%|██████████| 1953/1953 [17:41<00:00,  1.84it/s]  


In [6]:
gdf_with_stats

,CODIGO,OBSERV,INSUMO,APOYO,ASIGNACION,Asignado,OAM,area,url_type,download_url,poly_id,image_uid,geometry,r_mean,r_std,g_mean,g_std,b_mean,b_std
0,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,34429.374,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,0,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.97056 10.61903, -72.97054 ...",67.118672,63.050778,72.369456,65.615000,50.007945,48.514536
1,21,seasonal crops,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,6784.769,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.96914 10.61884, -72.96915 ...",86.838321,65.381449,76.577952,58.398020,63.137208,49.598628
2,112,urban not continuous,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,12227.436,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,2,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.96928 10.61948, -72.96931 ...",57.903879,54.918250,74.354503,66.705400,39.715053,44.734299
3,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,19385.187,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,3,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.97264 10.61976, -72.97273 ...",102.817986,53.979692,107.550803,51.629908,70.705621,42.725916
4,313,fragmented forest,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,5013.914,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,4,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.96917 10.62053, -72.9691 1...",32.369521,45.569768,42.636383,57.399349,21.679273,32.445704
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1948,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,1528.937,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1948,img_0029_37df798b,"MULTIPOLYGON (((-75.123 3.80901, -75.12297 3.8...",31.018763,45.725327,36.589655,52.510946,18.985246,29.199369
1949,333,Bare areas,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,0.026,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1949,img_0029_37df798b,"MULTIPOLYGON (((-75.12391 3.81075, -75.1239 3....",0.089548,2.602153,0.102208,2.904781,0.059827,1.799307
1950,232,wooded pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,10536.989,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1950,img_0029_37df798b,"MULTIPOLYGON (((-75.12278 3.8104, -75.12282 3....",30.101152,37.576581,33.759418,39.037403,19.319645,26.745836
1951,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,5139.198,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1951,img_0029_37df798b,"MULTIPOLYGON (((-75.12367 3.81079, -75.12357 3...",50.930026,41.676151,50.996732,40.269621,32.692476,29.041893


In [ ]:
gdf_with_stats.to_file(os.path.join(os.getcwd(), '..', 'data', 'geojson', 'polygon_cleaned_with_stats.geojson'))